<span style="font-weight:bold; font-size: 3rem; color:#333;">- Part 01: Feature Backfill for Train Delay Data (Two-Stage Model)</span>

## 🗒️ Overview

This notebook backfills historical features for the two-stage train delay model.

It performs the following steps:

1. **Choose train stations and time range**: define a list of LocationSignature codes for the Pendeltåg network and select a backfill window (e.g., last X days/months).
2. **Fetch historical TrainAnnouncement data** from Trafikverket's Open API using XML queries.
3. **Fetch auxiliary data** such as weather observations and ReasonCodes (if available) to enrich the dataset.
4. **Engineer features** such as delay minutes, time-of-day, day-of-week, recent delays, weather metrics, and reason categories.
5. **Save the resulting DataFrame** into a Hopsworks feature group for downstream training and inference.

> 🛠️ **Note**: You need to supply a valid Trafikverket API key. The API returns JSON when the request is sent in XML format. Replace placeholders where indicated.


### 📝 Imports

In [ ]:
import os
import datetime
import pandas as pd
import requests
import hopsworks_utils
import datetime as dt
from dotenv import load_dotenv
import hopsworks
from typing import Any, Dict, List, Optional, Tuple


# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

load_dotenv()


True

## 📡 Connect to Hopsworks Feature Store

In [ ]:
# Optional: Hopsworks storage (not required)
try:
    project = hopsworks_utils.HopsworksInterface()
    print("Hopsworks login OK")
except Exception as e:
    project = None
    print("Hopsworks not configured / login failed (OK). Proceeding without it.")
    print("Reason:", repr(e))


HOPSWORKS_API_KEY exists: True
HOPSWORKS_API_KEY length: 81
2026-01-05 16:54:08,411 INFO: Initializing external client
2026-01-05 16:54:08,415 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-01-05 16:54:09,659 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/2182
Hopsworks login OK


## 🔑 Configure Trafikverket API and Helper Functions

In [ ]:
def normalize_lists(df: pd.DataFrame) -> pd.DataFrame:
    """Convert list-columns to comma-separated strings."""
    if df.empty:
        return df

    list_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, list)).any()]
    if list_cols:
        print(f"Flattening list columns: {list_cols}")
        for c in list_cols:
            df[c] = df[c].apply(lambda x: ",".join(map(str, x)) if isinstance(x, list) else x)
    return df


In [ ]:


from zoneinfo import ZoneInfo

# ============================================================
# Trafikverket Open API (v2) helpers
# ============================================================

TRAFIKVERKET_BASE_URL = os.getenv(
    "TRAFIKVERKET_BASE_URL",
    "https://api.trafikinfo.trafikverket.se/v2/data.json",
)
API_KEY_TRAFIK = os.getenv("API_KEY_TRAFIK")

# Define Stockholm timezone for explicit conversion
STOCKHOLM_TZ = ZoneInfo("Europe/Stockholm")

def _iso(ts: dt.datetime) -> str:
    """
    Format timestamp as Trafikverket ISO string without timezone.
    Converts to Stockholm local time first, then formats as naive.
    """
    if ts.tzinfo is not None:
        ts = ts.astimezone(STOCKHOLM_TZ)
    return ts.strftime("%Y-%m-%dT%H:%M:%S")


def build_request_xml(
    api_key: str,
    object_type: str,
    filter_xml: str,
    include_fields: List[str],
    limit: int = 10000,
    schema_version: str = "1", # Weather uses v1 often, but v2 is available.
    order_by: Optional[str] = None,
) -> str:
    # 1. Format INCLUDE tags
    include_xml = "".join(f"<INCLUDE>{f}</INCLUDE>" for f in include_fields)

    # 2. Add 'orderby' as an attribute to the QUERY tag, NOT as a child element
    orderby_attr = f'orderby="{order_by}"' if order_by else ""

    return f"""
<REQUEST>
  <LOGIN authenticationkey="{api_key}" />
  <QUERY objecttype="{object_type}" schemaversion="{schema_version}" limit="{limit}" {orderby_attr}>
    {filter_xml}
    {include_xml}
  </QUERY>
</REQUEST>
""".strip()


def query_trafikverket(xml_body: str, timeout: int = 60) -> Dict[str, Any]:
    headers = {"Content-Type": "text/xml; charset=utf-8"}
    r = requests.post(
        TRAFIKVERKET_BASE_URL,
        data=xml_body.encode("utf-8"),
        headers=headers,
        timeout=timeout,
    )
    if not r.ok:
        print(f"❌ API Error {r.status_code}:")
        print(r.text)
    r.raise_for_status()
    return r.json()


def _extract_result_list(resp: Dict[str, Any], object_type: str) -> List[Dict[str, Any]]:
    """Return list of objects from RESPONSE.RESULT[0][object_type]."""
    try:
        return resp["RESPONSE"]["RESULT"][0].get(object_type, [])
    except Exception:
        return []


def _chunk_time_windows(start: dt.datetime, end: dt.datetime, hours: int) -> List[Tuple[dt.datetime, dt.datetime]]:
    out = []
    cur = start
    while cur < end:
        nxt = min(end, cur + dt.timedelta(hours=hours))
        out.append((cur, nxt))
        cur = nxt
    return out


# ============================================================
# Fetch: TrainAnnouncement (ops: timetable + actual + est + cancel)
# ============================================================

TRAINANNOUNCE_FIELDS = [
    "ActivityId",
    "ActivityType",                 # Arrival / Departure
    "AdvertisedTrainIdent",         # train number
    "AdvertisedTimeAtLocation",     # scheduled
    "EstimatedTimeAtLocation",      # predicted
    "TimeAtLocation",               # actual (when available)
    "LocationSignature",            # station code
    "Canceled",
    "Deleted",
    "InformationOwner",
    "Deviation",                    # sometimes contains cause info
    "FromLocation",
    "ToLocation",
    "TrackAtLocation",
]


def fetch_train_announcements(
    station_codes: List[str],
    start_time: dt.datetime,
    end_time: dt.datetime,
    window_hours: int = 6,
    limit: int = 10000,
    api_key: str = API_KEY_TRAFIK,
) -> pd.DataFrame:
    """
    Backfill TrainAnnouncement data by splitting time into windows.
    """
    all_rows: List[Dict[str, Any]] = []
    windows = _chunk_time_windows(start_time, end_time, hours=window_hours)

    # Indentation for XML readability
    station_or = "\n      ".join([f'<EQ name="LocationSignature" value="{s}" />' for s in station_codes])

    for (ws, we) in windows:
        filter_xml = f"""
<FILTER>
  <AND>
    <GT name="AdvertisedTimeAtLocation" value="{_iso(ws)}" />
    <LT name="AdvertisedTimeAtLocation" value="{_iso(we)}" />
    <OR>
      {station_or}
    </OR>
  </AND>
</FILTER>
""".strip()

        xml = build_request_xml(
            api_key=api_key,
            object_type="TrainAnnouncement",
            filter_xml=filter_xml,
            include_fields=TRAINANNOUNCE_FIELDS,
            limit=limit,
            order_by="AdvertisedTimeAtLocation",
        )
        resp = query_trafikverket(xml)
        rows = _extract_result_list(resp, "TrainAnnouncement")
        all_rows.extend(rows)

    if not all_rows:
        return pd.DataFrame()

    df = pd.json_normalize(all_rows)
    df = normalize_lists(df)
    
    

    
    # parse timestamps (some may be missing)
    for col in ["AdvertisedTimeAtLocation", "EstimatedTimeAtLocation", "TimeAtLocation"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # standardize names
    df = df.rename(
        columns={
            "AdvertisedTimeAtLocation": "scheduled_time",
            "EstimatedTimeAtLocation": "estimated_time",
            "TimeAtLocation": "actual_time",
            "LocationSignature": "station_code",
            "AdvertisedTrainIdent": "train_id",
        }
    )

    # choose best available "observed" time for historical delay calc
    df["observed_time"] = df["actual_time"].fillna(df["estimated_time"])

    # delay in minutes
    df["delay_min"] = (df["observed_time"] - df["scheduled_time"]).dt.total_seconds() / 60.0

    # event_time = scheduled_time for point-in-time alignment
    df["event_time"] = df["scheduled_time"]

    # cancellation flag normalize
    df["is_canceled"] = df.get("Canceled", False).fillna(False).astype(bool)

    # basic calendar fields
    df["hour"] = df["event_time"].dt.hour
    df["dow"] = df["event_time"].dt.dayofweek
    df["date"] = df["event_time"].dt.date

    # keep only the core columns we need downstream
    keep = [
        "ActivityId",
        "ActivityType",
        "train_id",
        "OperationalTrainNumber",
        "event_time",
        "scheduled_time",
        "estimated_time",
        "actual_time",
        "observed_time",
        "station_code",
        "delay_min",
        "is_canceled",
        "Deleted",
        "InformationOwner",
        "Deviation",
        "FromLocation",
        "ToLocation",
        "TrackAtLocation",
        "hour",
        "dow",
        "date",
    ]
    keep = [c for c in keep if c in df.columns]
    df = df[keep].sort_values(["train_id", "event_time", "station_code"]).reset_index(drop=True)
    return df


# ============================================================
# Fetch: TrainMessage (incident-ish, optional)
# ============================================================

TRAINMESSAGE_FIELDS = [
    "ExternalDescription",
    "ReasonCodeText",
    "StartDateTime",
    "LastUpdateDateTime",
    "AffectedLocation",
    "EventId",
]

def fetch_train_messages(
    start_time: dt.datetime,
    end_time: dt.datetime,
    limit: int = 10000,
    api_key: str = API_KEY_TRAFIK,
) -> pd.DataFrame:
    """Fetch TrainMessage in a time window."""

    # Simple filter on StartDateTime only
    filter_xml = f"""
<FILTER>
  <AND>
    <LT name="StartDateTime" value="{_iso(end_time)}" />
    <GT name="StartDateTime" value="{_iso(start_time)}" />
  </AND>
</FILTER>
""".strip()

    xml = build_request_xml(
        api_key=api_key,
        object_type="TrainMessage",
        filter_xml=filter_xml,
        include_fields=TRAINMESSAGE_FIELDS,
        limit=limit,
        order_by="StartDateTime",
    )
    resp = query_trafikverket(xml)
    rows = _extract_result_list(resp, "TrainMessage")
    if not rows:
        return pd.DataFrame()

    df = pd.json_normalize(rows)

    # Parse dates
    for col in ["StartDateTime", "LastUpdateDateTime", "CreatedDateTime", "LastModifiedDateTime"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


# ============================================================
# Weather: WeatherObservation (Limited retention ~7 days)
# ============================================================



# ============================================================
# Weather: WeatherObservation (Limited retention ~7 days)
# ============================================================

# CORRECTED FIELDS: 'MeasurementTime' is the correct field for time.
# We use Schema v2 notation (dot notation) for Temperature/Wind.
WEATHER_OBS_FIELDS = [
    "MeasurementTime",      # Fixed: was ObservationTime
    "ModifiedTime",
    "Id",
    "Air.Temperature",
    "Wind.Speed",
]

def fetch_weather_observations_last7d(
    start_time: dt.datetime,
    end_time: dt.datetime,
    limit: int = 10000,
    api_key: str = API_KEY_TRAFIK,
) -> pd.DataFrame:
    """
    WeatherObservation only keeps ~7 days historically.
    Returns empty if outside retention window.
    """
    filter_xml = f"""
<FILTER>
  <AND>
    <GT name="MeasurementTime" value="{_iso(start_time)}" />
    <LT name="MeasurementTime" value="{_iso(end_time)}" />
  </AND>
</FILTER>
""".strip()

    xml = build_request_xml(
        api_key=api_key,
        object_type="WeatherObservation",
        filter_xml=filter_xml,
        include_fields=WEATHER_OBS_FIELDS,
        limit=limit,
        order_by="MeasurementTime",
    )

    # We anticipate this might return empty for old dates, so we handle it gracefully.
    try:
        resp = query_trafikverket(xml)
        rows = _extract_result_list(resp, "WeatherObservation")
    except Exception as e:
        # If it fails (e.g. timeout or syntax), print but don't crash the whole pipeline
        print(f"⚠️ Weather fetch warning: {e}")
        return pd.DataFrame()

    if not rows:
        # Expected for 2023 dates (data purged after 7 days)
        return pd.DataFrame()

    df = pd.json_normalize(rows)
    for col in ["MeasurementTime", "ModifiedTime"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    df = df.rename(columns={"MeasurementTime": "weather_time"})
    return df

# ============================================================
# Join logic (ops ↔ incidents) + label creation
# ============================================================
def join_train_messages(
    ops_df: pd.DataFrame,
    msg_df: pd.DataFrame,
    time_buffer_min: int = 15,
) -> pd.DataFrame:
    """
    Best-effort incident join.

    We join TrainMessage onto stop events by station_code and time overlap.
    Since TrainMessage has no EndTime in the API, we assume it applies for 60 mins.

    Key improvement vs earlier version:
    - After merging, we deduplicate to ONE row per (train_id, event_time, station_code),
      keeping the latest *stop update* (ModifiedTime) and then the latest message (StartDateTime).
    """
    if ops_df.empty or msg_df.empty:
        ops_df = ops_df.copy()
        ops_df["reason_code"] = None
        ops_df["reason_text"] = None
        ops_df["reason_desc"] = None
        return ops_df

    m = msg_df.copy()

    # Flatten AffectedLocation
    affected_col = None
    for cand in ["AffectedLocation", "AffectedLocation.LocationSignature"]:
        if cand in m.columns:
            affected_col = cand
            break

    if affected_col is None:
        ops_df = ops_df.copy()
        ops_df["reason_code"] = None
        ops_df["reason_text"] = None
        ops_df["reason_desc"] = None
        return ops_df

    if affected_col == "AffectedLocation":
        m = m.explode("AffectedLocation")
        m["affected_station"] = m["AffectedLocation"].apply(
            lambda x: x.get("LocationSignature") if isinstance(x, dict) else None
        )
    else:
        m["affected_station"] = m[affected_col]

    m = m.dropna(subset=["affected_station"]).copy()

    # Create synthetic end time (Start + 1 hour)
    if "StartDateTime" in m.columns:
        m["StartDateTime"] = pd.to_datetime(m["StartDateTime"], errors="coerce")
    m["effective_end"] = m["StartDateTime"] + pd.Timedelta(hours=1)

    ops = ops_df.copy()

    # Ensure event_time is datetime
    ops["event_time"] = pd.to_datetime(ops["event_time"], errors="coerce")

    ops["t_start"] = ops["event_time"] - pd.to_timedelta(time_buffer_min, unit="m")
    ops["t_end"] = ops["event_time"] + pd.to_timedelta(time_buffer_min, unit="m")

    # Select columns to merge (only those that exist)
    cols_to_use = ["StartDateTime", "effective_end", "affected_station"]
    for c in ["ReasonCodeText", "ExternalDescription"]:
        if c in m.columns:
            cols_to_use.append(c)
    if "ReasonCode" in m.columns:
        cols_to_use.append("ReasonCode")

    merged = ops.merge(
        m[cols_to_use],
        left_on="station_code",
        right_on="affected_station",
        how="left",
    )

    # Keep rows with no message OR overlapping message window
    overlap = (
        merged["StartDateTime"].isna()
        | ((merged["StartDateTime"] <= merged["t_end"]) & (merged["effective_end"] >= merged["t_start"]))
    )
    merged = merged[overlap].copy()

    # ----------------------------
    # DEDUP (IMPROVED)
    # One row per stop event:
    #   (train_id, event_time, station_code)
    # Keep the latest stop update first (ModifiedTime),
    # then the latest message time (StartDateTime).
    # ----------------------------
    merged["msg_rank_time"] = merged["StartDateTime"].fillna(pd.Timestamp.min)

    if "ModifiedTime" in merged.columns:
        merged["ModifiedTime"] = pd.to_datetime(merged["ModifiedTime"], errors="coerce")
        merged["stop_rank_time"] = merged["ModifiedTime"].fillna(pd.Timestamp.min)
        merged = merged.sort_values(
            ["train_id", "event_time", "stop_rank_time", "msg_rank_time"],
            ascending=[True, True, False, False],
        )
    else:
        merged = merged.sort_values(
            ["train_id", "event_time", "msg_rank_time"],
            ascending=[True, True, False],
        )

    merged = merged.drop_duplicates(
        subset=["train_id", "event_time", "station_code"],
        keep="first",
    )

    # Rename / clean up
    merged = merged.rename(columns={
        "ReasonCodeText": "reason_text",
        "ExternalDescription": "reason_desc",
        "ReasonCode": "reason_code",
    })

    drop_cols = ["t_start", "t_end", "affected_station", "msg_rank_time", "effective_end", "stop_rank_time"]
    merged = merged.drop(columns=[c for c in drop_cols if c in merged.columns], errors="ignore")

    # Ensure columns exist even if message fields missing
    for col in ["reason_code", "reason_text", "reason_desc"]:
        if col not in merged.columns:
            merged[col] = None

    return merged


def add_labels(
    df: pd.DataFrame,
    horizon_min: int = 60,
    delay_threshold_min: int = 10,
) -> pd.DataFrame:
    """
    Create:
    - predictive label: will delay exceed threshold within next horizon?
    - reactive labels: final_delay for that train/day, additional_delay from now
    """
    if df.empty:
        return df

    out = df.copy()
    out["train_run_id"] = out["train_id"].astype(str) + "_" + out["date"].astype(str)

    # final delay per train run: use last known delay
    out["final_delay_min"] = out.groupby("train_run_id")["delay_min"].transform("last")
    out["additional_delay_min"] = out["final_delay_min"] - out["delay_min"]

    out = out.sort_values(["train_run_id", "event_time"]).reset_index(drop=True)
    horizon = pd.Timedelta(minutes=horizon_min)

    pred_flags = []
    for _, g in out.groupby("train_run_id", sort=False):
        times = g["event_time"].to_numpy()
        delays = g["delay_min"].to_numpy()
        y = []
        for i in range(len(g)):
            t0 = times[i]
            j = i
            max_d = -1e9
            while j < len(g) and (times[j] - t0) <= horizon:
                if pd.notna(delays[j]):
                    if delays[j] > max_d:
                        max_d = delays[j]
                j += 1
            y.append(1 if max_d >= delay_threshold_min else 0)
        pred_flags.extend(y)

    out["y_delay_within_horizon"] = pred_flags
    return out

In [ ]:
import re
import numpy as np
# ============================================================
# Fetch: TrainStation (static metadata)
# ============================================================

STATION_FIELDS = [
    "LocationSignature",
    "AdvertisedLocationName",
    "Geometry.WGS84",
]

def fetch_all_stations(api_key: str = API_KEY_TRAFIK) -> pd.DataFrame:
    """Fetch all train stations with geometry."""
    # We fetch ALL stations (empty filter) to ensure we get the coordinates
    xml = build_request_xml(
        api_key=api_key,
        object_type="TrainStation",
        filter_xml="", 
        include_fields=STATION_FIELDS,
    )
    
    resp = query_trafikverket(xml)
    rows = _extract_result_list(resp, "TrainStation")
    
    if not rows:
        return pd.DataFrame()
        
    df = pd.json_normalize(rows)
    return df

print("Fetching stations...")
stations_df = fetch_all_stations()
print(f"✅ Loaded {len(stations_df)} stations.")
def parse_wgs84_point(s):
    """
    Expects: "POINT (lon lat)"  e.g. "POINT (18.0687 59.3294)"
    """
    if not isinstance(s, str):
        return (np.nan, np.nan)
    m = re.search(r"POINT\s*\(\s*([0-9.\-]+)\s+([0-9.\-]+)\s*\)", s)
    if not m:
        return (np.nan, np.nan)
    lon = float(m.group(1))
    lat = float(m.group(2))
    return (lat, lon)

geom_col = "Geometry.WGS84"
if geom_col not in stations_df.columns:
    raise KeyError(f"'{geom_col}' not found in stations_df columns: {list(stations_df.columns)}")

stations_geo = stations_df.copy()
stations_geo[["lat", "lon"]] = stations_geo[geom_col].apply(lambda x: pd.Series(parse_wgs84_point(x)))
stations_geo = stations_geo.dropna(subset=["lat", "lon"])

display(stations_geo[["AdvertisedLocationName","LocationSignature","lat","lon"]].head(10))


Fetching stations...
✅ Loaded 1745 stations.


,AdvertisedLocationName,LocationSignature,lat,lon
0,Alingsås,A,57.926905,12.532185
1,Anneberg,Ag,57.538629,12.100776
2,Aneby,Any,57.837435,14.811896
3,Aspen,Apn,57.754422,12.240512
4,Arvika,Ar,59.653634,12.590803
5,Arboga,Arb,59.397189,15.840526
6,Arlanda C,Arnc,59.649072,17.928489
7,Aspedalen,Asd,57.762478,12.258478
8,Avesta Krylbo,Avky,60.129533,16.216148
9,Barkåkra,Baa,56.293656,12.824632


## 🕒 Define Backfill Parameters

In [ ]:
# Stockholm län bbox
MIN_LON, MIN_LAT = 17.25, 58.69
MAX_LON, MAX_LAT = 19.61, 60.27

stockholm_lan_df = stations_geo[
    (stations_geo["lon"] >= MIN_LON) & (stations_geo["lon"] <= MAX_LON) &
    (stations_geo["lat"] >= MIN_LAT) & (stations_geo["lat"] <= MAX_LAT)
].copy()

print("Stations in Stockholm län bbox:", len(stockholm_lan_df))

station_codes_stockholm_lan = (
    stockholm_lan_df["LocationSignature"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

station_codes_stockholm_lan[:30], len(station_codes_stockholm_lan)
station_codes = station_codes_stockholm_lan

Stations in Stockholm län bbox: 210


In [ ]:
# Increase time window for more trains
end_time = dt.datetime.now(dt.timezone.utc) + dt.timedelta(days=1)
start_time = end_time - dt.timedelta(days=30)

print("Start Time: ", start_time)

print("End Time: ", end_time)


#deth = 1/0
ops_df = fetch_train_announcements(
    station_codes=station_codes,
    start_time=start_time,
    end_time=end_time,
    window_hours=3,
    limit=50000,
)
print("RAW ops_df:", ops_df.shape)
print("Unique stop events (train_id,event_time,station_code):",
      ops_df.drop_duplicates(["train_id","event_time","station_code"]).shape)


print("ops rows:", len(ops_df))
display(ops_df.head())


Start Time:  2025-12-07 16:40:24.370592+00:00
End Time:  2026-01-06 16:40:24.370592+00:00
Flattening list columns: ['Deviation', 'FromLocation', 'ToLocation']
RAW ops_df: (96693, 20)
Unique stop events (train_id,event_time,station_code): (58856, 20)
ops rows: 96693


,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,is_canceled,Deleted,InformationOwner,Deviation,FromLocation,ToLocation,TrackAtLocation,hour,dow,date
0,1500adde-075d-66fb-08de-3de271d5a7e9,Avgang,10,2026-01-02 12:11:00+01:00,2026-01-02 12:11:00+01:00,NaT,NaT,NaT,Cst,NaN,True,False,SJ,"Inställt,Oväder",Cst,"U,Gä,Suc,Ös,Du",x,12,4,2026-01-02
1,1500adde-075d-66fb-08de-3de271d5a7ea,Ankomst,10,2026-01-02 12:33:00+01:00,2026-01-02 12:33:00+01:00,NaT,NaT,NaT,Arnc,NaN,True,False,SJ,"Inställt,Oväder",Cst,Du,x,12,4,2026-01-02
2,1500adde-075d-66fb-08de-3de271d5a7eb,Avgang,10,2026-01-02 12:34:00+01:00,2026-01-02 12:34:00+01:00,NaT,NaT,NaT,Arnc,NaN,True,False,SJ,"Inställt,Oväder",Cst,"U,Gä,Suc,Ös,Du",x,12,4,2026-01-02
3,1500adde-075d-66fb-08de-3de271d5a7ec,Ankomst,10,2026-01-02 12:54:00+01:00,2026-01-02 12:54:00+01:00,NaT,NaT,NaT,U,NaN,True,False,SJ,"Inställt,Oväder","Cst,Arnc",Du,x,12,4,2026-01-02
4,1500adde-075d-66fb-08de-3de271d809b2,Avgang,10,2026-01-02 12:55:00+01:00,2026-01-02 12:55:00+01:00,NaT,NaT,NaT,U,NaN,True,False,SJ,"Inställt,Oväder",Cst,"Gä,Shv,Suc,Ös,Du",x,12,4,2026-01-02


In [ ]:
#Sort ops_df by event_time descending and print the first few rows
ops_df = ops_df.sort_values(by="event_time", ascending=False)

display(ops_df.head())

,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,is_canceled,Deleted,InformationOwner,Deviation,FromLocation,ToLocation,TrackAtLocation,hour,dow,date
0,1500adde-075d-66fb-08de-411db402c32f,Ankomst,2754,2026-01-06 17:40:00+01:00,2026-01-06 17:40:00+01:00,NaT,NaT,NaT,Sst,NaN,False,False,SL,NaN,Söc,Mr,1,17,1,2026-01-06
1,1500adde-075d-66fb-08de-411d9ae9a525,Ankomst,2254,2026-01-06 17:40:00+01:00,2026-01-06 17:40:00+01:00,NaT,NaT,NaT,Sod,NaN,False,False,SL,NaN,Äs,U,1,17,1,2026-01-06
6,1500adde-075d-66fb-08de-411db8c2a6aa,Ankomst,2852,2026-01-06 17:40:00+01:00,2026-01-06 17:40:00+01:00,NaT,NaT,NaT,Kän,NaN,False,False,SL,NaN,Vhe,Kän,2,17,1,2026-01-06
2,1500adde-075d-66fb-08de-411d7381e439,Avgang,140,2026-01-06 17:40:00+01:00,2026-01-06 17:40:00+01:00,NaT,NaT,NaT,Söö,NaN,False,False,Mälardalstrafik AB,NaN,Hpbg,"Flb,Cst",6,17,1,2026-01-06
3,1500adde-075d-66fb-08de-411da6cd6cee,Avgang,2552,2026-01-06 17:40:00+01:00,2026-01-06 17:40:00+01:00,NaT,NaT,NaT,Spå,NaN,False,False,SL,Kort tåg,Nyc,Bål,3,17,1,2026-01-06


In [ ]:
msg_df = fetch_train_messages(start_time=start_time, end_time=end_time)
print("messages rows:", len(msg_df))
display(msg_df.head())


messages rows: 0


""


In [ ]:
xml = f"""
<REQUEST>
  <LOGIN authenticationkey="{API_KEY_TRAFIK}" />
  <QUERY objecttype="TrainMessage" schemaversion="1.7" limit="5">
    <FILTER />
    <INCLUDE>EventId</INCLUDE>
    <INCLUDE>StartDateTime</INCLUDE>
    <INCLUDE>Header</INCLUDE>
    <INCLUDE>ExternalDescription</INCLUDE>
    <INCLUDE>AffectedLocation</INCLUDE>
    <INCLUDE>LastUpdateDateTime</INCLUDE>
  </QUERY>
</REQUEST>
""".strip()

resp = query_trafikverket(xml)
print(resp)


{'RESPONSE': {'RESULT': [{'TrainMessage': []}]}}


In [ ]:
ops_df = join_train_messages(ops_df, msg_df)
print("ops rows after joining messages:", len(ops_df))
display(ops_df.head())


ops rows after joining messages: 76076


,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,...,Deviation,FromLocation,ToLocation,TrackAtLocation,hour,dow,date,reason_code,reason_text,reason_desc
0,1500adde-075d-66fb-08de-3de271d5a7e9,Avgang,10,2026-01-02 12:11:00+01:00,2026-01-02 12:11:00+01:00,NaT,NaT,NaT,Cst,NaN,...,"Inställt,Oväder",Cst,"U,Gä,Suc,Ös,Du",x,12,4,2026-01-02,None,None,None
1,1500adde-075d-66fb-08de-3de271d5a7ea,Ankomst,10,2026-01-02 12:33:00+01:00,2026-01-02 12:33:00+01:00,NaT,NaT,NaT,Arnc,NaN,...,"Inställt,Oväder",Cst,Du,x,12,4,2026-01-02,None,None,None
2,1500adde-075d-66fb-08de-3de271d5a7eb,Avgang,10,2026-01-02 12:34:00+01:00,2026-01-02 12:34:00+01:00,NaT,NaT,NaT,Arnc,NaN,...,"Inställt,Oväder",Cst,"U,Gä,Suc,Ös,Du",x,12,4,2026-01-02,None,None,None
3,1500adde-075d-66fb-08de-3de271d5a7ec,Ankomst,10,2026-01-02 12:54:00+01:00,2026-01-02 12:54:00+01:00,NaT,NaT,NaT,U,NaN,...,"Inställt,Oväder","Cst,Arnc",Du,x,12,4,2026-01-02,None,None,None
4,1500adde-075d-66fb-08de-3de271d809b2,Avgang,10,2026-01-02 12:55:00+01:00,2026-01-02 12:55:00+01:00,NaT,NaT,NaT,U,NaN,...,"Inställt,Oväder",Cst,"Gä,Shv,Suc,Ös,Du",x,12,4,2026-01-02,None,None,None


In [ ]:
import requests
import pandas as pd

def fetch_openmeteo_archive(lat: float, lon: float, start_date: str, end_date: str) -> pd.DataFrame:
    """
    start_date/end_date: 'YYYY-MM-DD'
    Returns hourly weather in Europe/Stockholm timezone.
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,precipitation,rain,snowfall,windspeed_10m",
        "timezone": "Europe/Stockholm",
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    j = r.json()

    if "hourly" not in j or "time" not in j["hourly"]:
        return pd.DataFrame()

    h = pd.DataFrame(j["hourly"])
    h["weather_time"] = pd.to_datetime(h["time"])
    h = h.drop(columns=["time"])
    return h


In [ ]:
# --- Labels (CREATE df) ---
df = add_labels(
    ops_df,
    horizon_min= 60,
    delay_threshold_min=10
)

print("df rows:", len(df))
#display(df.head())


df rows: 76076


In [ ]:
# Example: Stockholm (roughly)
weather_hourly = fetch_openmeteo_archive(
    lat=59.3293, lon=18.0686,
    start_date="2026-01-01", end_date="2026-01-04"
)
print(len(weather_hourly))
#display(weather_hourly.head())



HTTPError: 400 Client Error: Bad Request for url: https://archive-api.open-meteo.com/v1/archive?latitude=59.3293&longitude=18.0686&start_date=2026-01-01&end_date=2026-01-06&hourly=temperature_2m%2Cprecipitation%2Crain%2Csnowfall%2Cwindspeed_10m&timezone=Europe%2FStockholm

In [ ]:
# 1) Ensure datetime dtype + same timezone handling
df2 = df.copy()
w2  = weather_hourly.copy()

df2["event_time"] = pd.to_datetime(df2["event_time"], errors="coerce")
w2["weather_time"] = pd.to_datetime(w2["weather_time"], errors="coerce")

# If one is timezone-aware and the other isn't, normalize both to naive (or both to same tz)
if getattr(df2["event_time"].dt, "tz", None) is not None:
    df2["event_time"] = df2["event_time"].dt.tz_convert("Europe/Stockholm").dt.tz_localize(None)

if getattr(w2["weather_time"].dt, "tz", None) is not None:
    w2["weather_time"] = w2["weather_time"].dt.tz_convert("Europe/Stockholm").dt.tz_localize(None)

# 2) Drop rows with invalid timestamps
df2 = df2.dropna(subset=["event_time"]).copy()
w2  = w2.dropna(subset=["weather_time"]).copy()

# 3) Sort (required!)
df2 = df2.sort_values("event_time").reset_index(drop=True)
w2  = w2.sort_values("weather_time").reset_index(drop=True)

print(df2["event_time"].dtype, w2["weather_time"].dtype)
print("df2 rows:", len(df2), "w2 rows:", len(w2))
print("event_time range:", df2["event_time"].min(), "->", df2["event_time"].max())
print("weather_time range:", w2["weather_time"].min(), "->", w2["weather_time"].max())




df2 = pd.merge_asof(
    df2,
    w2,
    left_on="event_time",
    right_on="weather_time",
    direction="nearest",
    tolerance=pd.Timedelta("1H"),
)
display(df2.head())
df=df2


datetime64[ns] datetime64[ns]
df2 rows: 76076 w2 rows: 744
event_time range: 2026-01-02 00:00:00 -> 2026-01-05 16:54:00
weather_time range: 2025-12-05 00:00:00 -> 2026-01-04 23:00:00


,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,...,train_run_id,final_delay_min,additional_delay_min,y_delay_within_horizon,temperature_2m,precipitation,rain,snowfall,windspeed_10m,weather_time
0,1500adde-075d-66fb-08de-3de2e9c51bd7,Avgang,2983,2026-01-02 00:00:00,2026-01-02 00:00:00+01:00,NaT,2026-01-02 00:00:00+01:00,2026-01-02 00:00:00+01:00,Mr,0.0,...,2983_2026-01-02,0.0,0.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
1,1500adde-075d-66fb-08de-3de2c860a960,Avgang,2477,2026-01-02 00:03:00,2026-01-02 00:03:00+01:00,NaT,2026-01-02 00:03:00+01:00,2026-01-02 00:03:00+01:00,Söc,0.0,...,2477_2026-01-02,-2.0,-2.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
2,1500adde-075d-66fb-08de-3de2e9c6a56c,Avgang,2983,2026-01-02 00:04:00,2026-01-02 00:04:00+01:00,NaT,2026-01-02 00:05:00+01:00,2026-01-02 00:05:00+01:00,Rs,1.0,...,2983_2026-01-02,0.0,-1.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
3,1500adde-075d-66fb-08de-3de2e9c6a56b,Ankomst,2983,2026-01-02 00:04:00,2026-01-02 00:04:00+01:00,NaT,2026-01-02 00:04:00+01:00,2026-01-02 00:04:00+01:00,Rs,0.0,...,2983_2026-01-02,0.0,0.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
4,1500adde-075d-66fb-08de-3de314d64fcf,Avgang,7879,2026-01-02 00:05:00,2026-01-02 00:05:00+01:00,NaT,2026-01-02 00:05:00+01:00,2026-01-02 00:05:00+01:00,Arnn,0.0,...,7879_2026-01-02,1.0,1.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02


In [ ]:
display(df.head())

,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,...,train_run_id,final_delay_min,additional_delay_min,y_delay_within_horizon,temperature_2m,precipitation,rain,snowfall,windspeed_10m,weather_time
0,1500adde-075d-66fb-08de-3de2e9c51bd7,Avgang,2983,2026-01-02 00:00:00,2026-01-02 00:00:00+01:00,NaT,2026-01-02 00:00:00+01:00,2026-01-02 00:00:00+01:00,Mr,0.0,...,2983_2026-01-02,0.0,0.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
1,1500adde-075d-66fb-08de-3de2c860a960,Avgang,2477,2026-01-02 00:03:00,2026-01-02 00:03:00+01:00,NaT,2026-01-02 00:03:00+01:00,2026-01-02 00:03:00+01:00,Söc,0.0,...,2477_2026-01-02,-2.0,-2.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
2,1500adde-075d-66fb-08de-3de2e9c6a56c,Avgang,2983,2026-01-02 00:04:00,2026-01-02 00:04:00+01:00,NaT,2026-01-02 00:05:00+01:00,2026-01-02 00:05:00+01:00,Rs,1.0,...,2983_2026-01-02,0.0,-1.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
3,1500adde-075d-66fb-08de-3de2e9c6a56b,Ankomst,2983,2026-01-02 00:04:00,2026-01-02 00:04:00+01:00,NaT,2026-01-02 00:04:00+01:00,2026-01-02 00:04:00+01:00,Rs,0.0,...,2983_2026-01-02,0.0,0.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
4,1500adde-075d-66fb-08de-3de314d64fcf,Avgang,7879,2026-01-02 00:05:00,2026-01-02 00:05:00+01:00,NaT,2026-01-02 00:05:00+01:00,2026-01-02 00:05:00+01:00,Arnn,0.0,...,7879_2026-01-02,1.0,1.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02


## 🧬 Create Feature Group and Insert Historical Data

In [ ]:
import os, datetime as dt

out_path = "data/train_stop_events_labeled.parquet"
out_path_weather = "data/weather_features.parquet"
out_path_station = "data/station_features.parquet"


os.makedirs(os.path.dirname(out_path), exist_ok=True)

if "df" not in globals() or df is None or df.empty:
    print("No labeled data produced; nothing to save.")
else:
    df.to_parquet(out_path, index=False)
    stations_geo.to_parquet(out_path_station, index=False)

    # weather_df may not exist in your current run — save only if present
    if "weather_df" in globals() and weather_df is not None and not weather_df.empty:
        weather_df.to_parquet(out_path_weather, index=False)

    ts = os.path.getmtime(out_path)
    print("✅ Saved canonical labeled table:", out_path)
    print("File last modified:", dt.datetime.fromtimestamp(ts))


✅ Saved canonical labeled table: data/train_stop_events_labeled.parquet
File last modified: 2026-01-05 16:55:36.017768
